[<- Z3-Python-10](Z3-10-Cryptarithmetic-Python.ipynb) | [README Z3-Python](README.md) | [Z3-Python-12 ->](Z3-12-Real-Arithmetic-Python.ipynb)

# 11 - Coloration de Graphe avec Z3 (Rust)

## Navigation

- [Index](../README.md)
- Notebook précédent : [Z3-Python-10 Cryptarithmes](Z3-10-Cryptarithmetic-Python.ipynb)
- Notebook suivant : [Z3-Python-12 Real Arithmetic](Z3-12-Real-Arithmetic-Python.ipynb)
- Famille : [SymbolicAI/SMT/Z3-API](../../README.md)
- Jumeau .NET : [12_Graph_Coloring_Petersen (jumeau .NET)](../Z3-Linq2Z3/12_Graph_Coloring_Petersen.ipynb)

***

## Objectif pédagogique

A la fin de ce notebook, vous saurez :

1. **Modeliser un Problème de coloration de graphe** avec Z3 (variables entières, contraintes différentes, recherche linéaire).
2. **Resoudre avec un solveur SMT** : `Solver()` + `And`/`Or`/`Distinct`/`Implies` et recherche de modèles.
3. **Prouver l'optimalite** : trouver le plus petit k tel que le graphe soit k-coloriable (le **nombre chromatique** chi(G)).
4. **Comparer heuristique et SMT** : le glouton first-fit est rapide mais peut donner chi+1 ; le solveur prouve l'optimalite.
5. **Generaliser** : variantes (clique forcee, planification d'examens, comptage de colorations).

## Plan du notebook

| Section | Contenu | Cellules clés |
|---------|---------|---------------|
| 1. Graphe de Petersen | 10 sommets, 15 aretes, structure symétrique | code[1] |
| 2. Heuristique gloutonne | first-fit (rapide, sous-optimal) | code[2] |
| 3. Modèle Z3 | 1 variable entière par sommet, contraintes différentes | code[3] |
| 4. Nombre chromatique | recherche linéaire k=1..N | code[4], chi(Petersen)=3 |
| 5. Glouton vs solveur | 3 leçons de pedagogie | section 5 |
| 6. Exercices | K4 force, planification, comptage | code[18, 20, 22] |
| 7. Conclusion | Heuristique vs preuve | section 7 |

## Prerequis

- **Notebooks** : Z3-Python-01 (Setup), Z3-Python-02 (Logique booleenne), Z3-Python-05 (Quantificateurs).
- **Bibliotheques** : la crate `z3` (solveur SMT Microsoft Research) ; `matplotlib` n'est pas portee dans ce lot, la visualisation devient un rendu texte.
- **Rust** : kernelspec `rust` (evcxr), une seule directive `:dep` en tete de carnet.
- **Maths** : théorie des graphes (cliques, chi, théorème des 4 couleurs), NP-completude de 3-COLOR.

## Verdict SOTA

Ce notebook utilise **Z3** (le solveur SMT de Microsoft Research, leader du concours SMT-COMP pour la théorie QF_LIA) -- le moteur SOTA pour la verification de contraintes lineaires entières. Pour des colorations de graphes a grande echelle (> 10^5 sommets), on peut utiliser **OR-Tools CP-SAT** (Google, cf CSP-8-Temporal) ou des algorithme approches (DSATUR, Welsh-Powell).

## Verbatim code[0] : setup

Sortie verbatim : `Imports OK : z3` -- la ligne est adaptée (matplotlib n'est pas portee dans ce lot), elle n'est donc pas comparable a celle du jumeau Python.

In [1]:
:dep z3 = { version = "0.24.0", features = ["gh-release"] }

use z3::{ast::{Ast, Bool, Int}, SatResult, Solver};

// mode batch : inutile en Rust, aucune figure n'est produite dans ce lot
println!("Imports OK : z3");

Imports OK : z3


## 1. Le graphe de Petersen

Le graphe de Petersen a **10 sommets (0 a 9)** et **15 aretes** organisees en :
- Un **pentagone externe** (sommets 0 a 4)
- Un **pentagone interne** (sommets 5 a 9, formant un pentagramme)
- 5 **rayons** connectant chaque sommet externe a son correspondant interne

### Proprietes remarquables

- **3-regulier** : chaque sommet a exactement 3 voisins
- **Graphe de Moore** : 10 sommets, 15 aretes, diametre 2
- **Non-hamiltonien** : aucun cycle passant par tous les sommets
- **Non-planaire** : ne peut pas etre dessine sans croisements d'aretes
- **3-coloriable** mais pas 2-coloriable (contient des cycles impairs)
- **Nombre chromatique chi = 3** (clairement non trivial)
- **Graphe symétrique** : 120 automorphismes (groupe sym(5))

### Verbatim code[1] : definition

Sortie verbatim : graphe de Petersen défini avec 10 sommets et 15 aretes, organisees en pentagone externe + pentagramme interne + rayons.

### Cas d'usage

- **Cartographie de conflits** : sommets = personnes ou pays, aretes = conflits (coloration = groupes de paix).
- **Planification** : sommets = examens, aretes = étudiants communs (cf Exercice 2).
- **Allocation de frequences** : sommets = antennes, aretes = interference, couleurs = frequences.
- **Compilation de registres** : variables = temporaires, aretes = interference, couleurs = registres physiques (NP-complet).

In [2]:
// Graphe de Petersen : 10 sommets (0..9), 15 aretes
const N: usize = 10;

let edges: Vec<(usize, usize)> = vec![
    // Pentagone externe
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
    // Rayons externe -> interne
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
    // Pentagramme interne
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),
];

// Liste d'adjacence (pour le glouton)
let mut adj: Vec<Vec<usize>> = vec![Vec::new(); N];
for (a, b) in &edges {
    adj[*a].push(*b);
    adj[*b].push(*a);
}
let _ = &adj; // consommee par le glouton dans la cellule suivante

fn display_coloring(color: &[usize], title: &str, edges: &[(usize, usize)]) {
    println!("{}", title);
    let k = color.iter().copied().max().map_or(0, |m| m + 1);
    println!("  Couleurs utilisees : {}", k);
    for (v, c) in color.iter().enumerate() {
        println!("  Sommet {} -> couleur {}", v, c);
    }
    let valid = edges.iter().all(|(a, b)| color[*a] != color[*b]);
    println!("  Coloration valide (aucune arete monochrome) : {}", if valid { "True" } else { "False" });
}

println!("Graphe de Petersen charge : {} sommets, {} aretes.", N, edges.len());

Graphe de Petersen charge : 10 sommets, 15 aretes.


### Verbatim code[1] : definition du graphe

Sortie verbatim : definition de N=10 sommets, edges=15 aretes (5 pentagone externe + 5 pentagramme interne + 5 rayons).

### Lecture approfondie — la liste d'aretes du Petersen

La liste des aretes est explicite :
- **Pentagone externe** : (0,1), (1,2), (2,3), (3,4), (4,0)
- **Rayons** : (0,5), (1,6), (2,7), (3,8), (4,9)
- **Pentagramme interne** : (5,7), (7,9), (9,6), (6,8), (8,5) -- connexion par sauts de 2

### Verification

On peut vérifier le degré de chaque sommet (tous = 3) :
```rust
let mut deg: Vec<usize> = vec![0; N];
for (u, v) in &edges {
    deg[*u] += 1;
    deg[*v] += 1;
}
println!("Degres : {:?}", deg); // tous = 3
```

### Symétries

Le graphe de Petersen a **120 automorphismes** (le groupe sym(5)). On peut vérifier que les automorphismes preservent les degrés et la structure.

## 2. Approche 1 : heuristique gloutonne (first-fit)

L'heuristique **first-fit** parcourt les sommets dans un ordre donne et attribue a chaque sommet la plus petite couleur qui n'est pas utilisee par ses voisins déjà colores.

### Algorithme

```rust
fn greedy_coloring(n: usize, adjacency: &[Vec<usize>], order: &[usize]) -> Vec<i64> {
    let mut color: Vec<i64> = vec![-1; n]; // -1 : sommet non encore colorie
    let mut forbidden_at: Vec<usize> = vec![0; n]; // horodatage des couleurs interdites
    for (step, &v) in order.iter().enumerate() {
        let stamp = step + 1;
        for &w in &adjacency[v] {
            if color[w] >= 0 {
                forbidden_at[color[w] as usize] = stamp;
            }
        }
        let mut c: i64 = 0;
        while forbidden_at[c as usize] == stamp {
            c += 1;
        }
        color[v] = c;
    }
    color
}
```

Le tableau `forbidden_at` marque les couleurs interdites avec l'horodatage du sommet courant. Chaque voisin n'est ainsi inspecte qu'une fois, sans recherche lineaire dans une collection de couleurs.

### Complexite

L'implementation inspecte chaque sommet et chaque arete un nombre constant de fois : **O(n + m)** en temps, avec n = sommets et m = aretes, et **O(n)** en memoire auxiliaire. Elle reste donc tres rapide pour de tres grands graphes (> 10^6 sommets).

### Qualite

Le glouton **ne garantit pas l'optimalite**. Pour le graphe de Petersen, l'ordre naturel des sommets (0..9) lui donne **3 couleurs** (l'optimum) ; c'est un ordre de parcours **défavorable** choisi explicitement dans code[2] qui le fait monter à **4** -- l'algorithme ne trie les sommets ni par degré ni par identifiant, il suit l'ordre qu'on lui donne.

### Verbatim code[2]

Sortie verbatim : `First-fit, ordre naturel 0..9 :` puis `Couleurs utilisees : 3` ; `First-fit, ordre defavorable [4, 8, 2, 6, 5, 9, 0, 7, 1, 3] :` puis `Couleurs utilisees : 4` ; et la ligne de synthese `Meme graphe, meme algorithme : 3 couleurs vs 4 couleurs selon l'ordre.`

### Cas d'usage industriel

- **Allocation de frequences** dans les réseaux mobiles : jusqu'a 10^4 antennes, glouton rapide, solutions a 1-2 couleurs de l'optimum.
- **Coloration de graphes d'interferences** : exécution en temps linéaire.

### Optimalite pour certains graphes

Le glouton est **optimal** pour les graphes bipartis (chi=2) et les graphes d'intervalles (chi = taille de la plus grande clique). Pour les graphes généraux, il peut donner chi + 1 voire plus.

In [3]:
// Coloration gloutonne first-fit selon un ordre de parcours donne
const N: usize = 10;

let edges: Vec<(usize, usize)> = vec![
    // Pentagone externe
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
    // Rayons externe -> interne
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
    // Pentagramme interne
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),
];

// Liste d'adjacence (pour le glouton)
let mut adj: Vec<Vec<usize>> = vec![Vec::new(); N];
for (a, b) in &edges {
    adj[*a].push(*b);
    adj[*b].push(*a);
}

fn greedy_coloring(n: usize, adjacency: &[Vec<usize>], order: &[usize]) -> Vec<i64> {
    let mut color: Vec<i64> = vec![-1; n]; // -1 : sommet non encore colorie
    let mut forbidden_at: Vec<usize> = vec![0; n];
    for (step, &v) in order.iter().enumerate() {
        let stamp = step + 1;
        for &w in &adjacency[v] {
            if color[w] >= 0 {
                forbidden_at[color[w] as usize] = stamp;
            }
        }
        let mut c: i64 = 0;
        while forbidden_at[c as usize] == stamp {
            c += 1;
        }
        color[v] = c;
    }
    color
}

fn display_coloring(color: &[usize], title: &str, edges: &[(usize, usize)]) {
    println!("{}", title);
    let k = color.iter().copied().max().map_or(0, |m| m + 1);
    println!("  Couleurs utilisees : {}", k);
    for (v, c) in color.iter().enumerate() {
        println!("  Sommet {} -> couleur {}", v, c);
    }
    let valid = edges.iter().all(|(a, b)| color[*a] != color[*b]);
    println!("  Coloration valide (aucune arete monochrome) : {}", if valid { "True" } else { "False" });
}

// Ordre naturel 0..9
let natural_order: Vec<usize> = (0..N).collect();
let greedy_natural: Vec<i64> = greedy_coloring(N, &adj, &natural_order);
let greedy_natural_colors: i64 = greedy_natural.iter().copied().max().map_or(0, |m| m + 1);
let natural_colors: Vec<usize> = greedy_natural.iter().map(|c| *c as usize).collect();
display_coloring(&natural_colors, "First-fit, ordre naturel 0..9 :", &edges);
println!();

// Ordre defavorable (force la 4e couleur)
let bad_order: Vec<usize> = vec![4, 8, 2, 6, 5, 9, 0, 7, 1, 3];
let greedy_bad: Vec<i64> = greedy_coloring(N, &adj, &bad_order);
let greedy_bad_colors: i64 = greedy_bad.iter().copied().max().map_or(0, |m| m + 1);
let bad_colors: Vec<usize> = greedy_bad.iter().map(|c| *c as usize).collect();
let bad_title = format!("First-fit, ordre defavorable {:?} :", bad_order);
display_coloring(&bad_colors, &bad_title, &edges);
println!();
println!("Meme graphe, meme algorithme : {} couleurs vs {} couleurs selon l'ordre.", greedy_natural_colors, greedy_bad_colors);

First-fit, ordre naturel 0..9 :


  Couleurs utilisees : 3


  Sommet 0 -> couleur 0


  Sommet 1 -> couleur 1


  Sommet 2 -> couleur 0


  Sommet 3 -> couleur 1


... (17 outputs left out)

  Sommet 8 -> couleur 0


  Sommet 9 -> couleur 2


  Coloration valide (aucune arete monochrome) : True


Meme graphe, meme algorithme : 3 couleurs vs 4 couleurs selon l'ordre.


## Interpretation

Le glouton trouve **3 couleurs** dans l'ordre naturel (0..9) mais **4 dans l'ordre défavorable** `[4, 8, 2, 6, 5, 9, 0, 7, 1, 3]` -- les deux sorties de code[2] sont côte à côte.

### Lecture — l'ordre de parcours fait varier le glouton

Cela illustre une propriete cle : **l'ordre de parcours** peut modifier significativement le résultat du glouton. Pour un même graphe, l'ordre de parcours peut faire varier le nombre de couleurs de chi(G) a chi(G) + 1, voire plus pour certains graphes.

### Verdict

L'heuristique gloutonne est **rapide** (O(n+m)) mais **non garantie optimale**. Pour obtenir le nombre chromatique exact, il faut :
- Backtracking avec propagation (exact, exponentiel dans le pire cas).
- Solveur SMT/CSP (Z3, OR-Tools CP-SAT) -- optimal pour les instances moyennes.
- Recherche linéaire k=1..N : essayer chaque k et vérifier si k-coloriable.

### Cout computationnel

Pour le graphe de Petersen (n=10, m=15), le coût est celui du parcours O(n + m) -- la sortie n'affiche pas de mesure de temps. Pour les graphes industriels (n=10^5), il reste rapide -- **mais la verification d'optimalite necessite le solveur**.

## 3. Approche 2 : modelisation Z3

On encode une **variable entière** `C_v` par sommet (couleur de v). Pour chaque arete (u, v), on ajoute la contrainte `C_u != C_v` (les couleurs des voisins sont différentes).

### Implementation

```rust
for k in 1..=N {
    let s = Solver::new();
    for v in 0..N {
        s.assert(&color[v].ge(&Int::from_i64(0)));
        s.assert(&color[v].lt(&Int::from_i64(k as i64)));
    }
    // Brisure de symetrie : la couleur du sommet 0 est fixee
    s.assert(&color[0].eq(&Int::from_i64(0)));
    // Aretes : extremites de couleurs differentes
    for (a, b) in &edges {
        s.assert(&color[*a].ne(&color[*b]));
    }
    if s.check() == SatResult::Sat {
        let m = s.get_model(); // Option<Model> : aucun unwrap() ici
        // optimal_color = model.eval(&color[v], true).and_then(|val| val.as_i64())
        chromatic = k;
        break;
    }
}
```

### Verbatim code[3]

Sortie verbatim : `Modele defini : 10 variables de couleur entieres C0..C9.`

### Cout computationnel

Le solveur Z3 explore l'espace de recherche en utilisant DPLL(T) + propagation de clauses + théorie linéaire. Pour n=10 et k=3, c'est quasi-instantané.

### Avantage vs glouton

Le solveur fournit une **preuve d'optimalite** : si Z3 dit `unsat` pour k=2, alors **il n'existe aucune 2-coloration**. Le glouton ne peut pas fournir cette garantie.

### Variantes

- **Avec symétries** : ajouter une contrainte de symétrie (couleur 0 fixee pour le sommet 0) -- divise l'espace de recherche par k!.
- **Avec bornes explicites** : `Implies(Color[0] == 0, Or([Color[v] == 1 for v in adj[0]]))` pour forcer l'exploration.
- **Multi-shot** : `s.push()` / `s.pop(1)` pour explorer différentes hypotheses sans reconstruire le solveur.

In [4]:
// Modele Z3 : une variable de couleur par sommet
use z3::ast::{Ast, Int};

const N: usize = 10;

// Graphe de Petersen : 10 sommets (0..9), 15 aretes
let edges: Vec<(usize, usize)> = vec![
    // Pentagone externe
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
    // Rayons externe -> interne
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
    // Pentagramme interne
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),
];

// Liste d'adjacence complete, reproduite pour que le graphe soit autonome
let mut adj: Vec<Vec<usize>> = vec![Vec::new(); N];
for (a, b) in &edges {
    adj[*a].push(*b);
    adj[*b].push(*a);
}
let _ = &adj;

// Une variable entiere de couleur par sommet, nommees C0..C9
let color: Vec<Int> = (0..N).map(|v| Int::new_const(format!("C{v}"))).collect();

println!("Modele defini : {} variables de couleur entieres C0..C9.", N);

Modele defini : 10 variables de couleur entieres C0..C9.


## 4. Recherche du nombre chromatique

On cherche le plus petit `k` tel que le graphe soit `k`-coloriable. C'est le **nombre chromatique chi(G)**.

### Algorithme : recherche linéaire k=1..N

```rust
for k in 1..=N {
    let s = Solver::new();
    for v in 0..N {
        s.assert(&color[v].ge(&Int::from_i64(0)));
        s.assert(&color[v].lt(&Int::from_i64(k as i64)));
    }
    // Brisure de symetrie : la couleur du sommet 0 est fixee
    s.assert(&color[0].eq(&Int::from_i64(0)));
    // Aretes : extremites de couleurs differentes
    for (a, b) in &edges {
        s.assert(&color[*a].ne(&color[*b]));
    }
    match s.check() {
        SatResult::Unsat => println!("k = {} -> UNSAT", k),
        SatResult::Sat => {
            // m = s.get_model() : Option<Model>, valeurs via model.eval(&color[v], true)
            chromatic = k;
            break;
        }
        SatResult::Unknown => {
            // Un verdict inconnu ne prouve ni satisfaisabilite ni impossibilite.
            break;
        }
    }
}
```

Le premier résultat `SAT` prouve l'optimum **uniquement si chaque `k` inférieur a reçu un verdict `UNSAT`**. Un résultat `UNKNOWN` interrompt la recherche : sans les verdicts `UNSAT` pour toutes les valeurs inférieures, aucun nombre chromatique minimal ne peut être affirmé.

### Verbatim code[4]

Sortie verbatim : `k = 1 -> UNSAT`, `k = 2 -> UNSAT`, `k = 3 -> SAT : nombre chromatique trouve !` puis `Nombre chromatique chi(Petersen) = 3`.

### Cout computationnel

Pour le graphe de Petersen, on essaie k=1 (UNSAT), k=2 (UNSAT), k=3 (SAT) -- le total reste negligeable. Pour des graphes plus durs (Mycielski, Kneser), la recherche peut prendre plusieurs secondes.

### Optimalite

La recherche linéaire est **exacte** : le premier k qui donne SAT est **le nombre chromatique chi(G)** si tous les k précédents ont été déclarés `UNSAT`. Un verdict `UNKNOWN` ne permet pas de conclure et doit interrompre la recherche. On peut aussi utiliser une **recherche binaire** sur k (entre 1 et la borne superieure) pour accelerer.

### Variante : recherche binaire

```rust
let (mut lo, mut hi) = (1usize, N);
while lo < hi {
    let mid = (lo + hi) / 2;
    if is_k_coloriable(mid) {
        hi = mid;
    } else {
        lo = mid + 1;
    }
}
let chromatic = lo;
```

Logarithme de fois moins d'iterations (log_2(N) au lieu de N), mais chacune demande un appel SMT. Comme dans la recherche linéaire, un verdict `UNKNOWN` doit empêcher toute affirmation sur l'optimum.

In [5]:
// Recherche lineaire du nombre chromatique
use z3::{ast::{Ast, Bool, Int}, SatResult, Solver};

const N: usize = 10;

// Graphe de Petersen : 10 sommets (0..9), 15 aretes
let edges: Vec<(usize, usize)> = vec![
    // Pentagone externe
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
    // Rayons externe -> interne
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
    // Pentagramme interne
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),
];

// Liste d'adjacence complete, reproduite pour que le graphe soit autonome
let mut adj: Vec<Vec<usize>> = vec![Vec::new(); N];
for (a, b) in &edges {
    adj[*a].push(*b);
    adj[*b].push(*a);
}
let _ = &adj;

// Une variable entiere de couleur par sommet, nommees C0..C9
let color: Vec<Int> = (0..N).map(|v| Int::new_const(format!("C{v}"))).collect();

let mut optimal_color: Option<Vec<usize>> = None;
let mut chromatic: Option<usize> = None;
let mut interrupted = false;

for k in 1..=N {
    let k_i64 = k as i64;
    let s = Solver::new();
    // Domaine : couleur entieres entre 0 et k-1
    for v in 0..N {
        s.assert(&color[v].ge(&Int::from_i64(0)));
        s.assert(&color[v].lt(&Int::from_i64(k_i64)));
    }
    // Brisure de symetrie : la couleur du sommet 0 est fixee
    s.assert(&color[0].eq(&Int::from_i64(0)));
    // Aretes : extremites de couleurs differentes
    for (a, b) in &edges {
        s.assert(&color[*a].ne(&color[*b]));
    }

    match s.check() {
        SatResult::Unsat => {
            println!("k = {} -> UNSAT (impossible avec {} couleur(s))", k, k);
        }
        SatResult::Unknown => {
            println!("k = {} -> UNKNOWN : Z3 n'a pas decide, recherche interrompue.", k);
            interrupted = true;
            break;
        }
        SatResult::Sat => {
            let m = match s.get_model() {
                Some(m) => m,
                None => {
                    println!("k = {} -> SAT mais aucun modele disponible : recherche interrompue.", k);
                    interrupted = true;
                    break;
                }
            };

            let mut sol: Vec<usize> = Vec::with_capacity(N);
            let mut complete = true;
            for v in 0..N {
                match m.eval(&color[v], true).and_then(|val| val.as_i64()) {
                    Some(c) => sol.push(c as usize),
                    None => {
                        complete = false;
                        break;
                    }
                }
            }
            if !complete {
                println!("Modele incomplet : couleur de certains sommets non entieres, recherche interrompue.");
                interrupted = true;
                break;
            }

            optimal_color = Some(sol);
            chromatic = Some(k);
            println!("k = {} -> SAT : nombre chromatique trouve !", k);
            break;
        }
    }
}

println!();
if let Some(k) = chromatic {
    println!("Nombre chromatique chi(Petersen) = {}", k);
} else if interrupted {
    println!("Nombre chromatique non determine : la preuve de minimalite est incomplete.");
};

k = 1 -> UNSAT (impossible avec 1 couleur(s))


k = 2 -> UNSAT (impossible avec 2 couleur(s))


k = 3 -> SAT : nombre chromatique trouve !


Nombre chromatique chi(Petersen) = 3


## Interpretation : `chi = 3` prouve

Z3 confirme **`chi(Petersen) = 3`** : `k=1` et `k=2` sont `unsat` (aucune 1-coloration ni 2-coloration), `k=3` est `sat`.

### Verbatim code[4] : recherche linéaire

Sortie verbatim de la cellule de recherche : `Nombre chromatique chi(Petersen) = 3`. La coloration sommet par sommet correspondante est extraite et affichée dans la cellule suivante.

### Lecture approfondie — le verdict k=1, k=2, k=3 du solveur

Pour k=1, Z3 retourne UNSAT (UNSATISFIABLE) : impossible de colorier avec 1 seule couleur car il y a des aretes.

Pour k=2, Z3 retourne UNSAT : Petersen contient des cycles impairs (5-cycle dans le pentagone externe), donc non 2-coloriable.

Pour k=3, Z3 retourne SAT : le Petersen admet plusieurs 3-colorations valides. Le témoin affiche etablit la faisabilite pour k = 3, tandis que les verdicts UNSAT pour k = 1 et k = 2 etablissent la minimalite de 3 couleurs.

Attention : une suite reguliere comme `[0, 1, 2, 0, 1, 2, 0, 1, 2, 0]` n'est **pas** une coloration valide du Petersen -- elle laisse les aretes (9, 6) et (8, 5) monochromes (verifiable sur les 15 aretes).

### Cout computationnel

3 appels au solveur (k=1, k=2, k=3), chacun sur 10 variables entieres et 15 contraintes de difference -- la sortie n'affiche pas de mesure de temps.

### Optimalite

La recherche lineaire est exacte : le premier k qui donne SAT est **le nombre chromatique chi(G)** à condition que tous les k précédents aient reçu le verdict UNSAT. Un verdict UNKNOWN interrompt la recherche et ne permet pas de conclure.

### Verification manuelle

La 3-coloration peut-être verifiee a la main : pour chaque arete (u, v), C_u != C_v. La liste des 15 aretes est verifiee en 15 operations.

### Variante

Recherche binaire entre 1 et N pour eviter d'essayer tous les k. Mais pour N=10, la recherche lineaire est suffisante.

### Verbatim code[5]

Sortie verbatim :
```
=== Comparaison ===
  Glouton first-fit (ordre naturel)    : 3 couleurs
  Glouton first-fit (ordre defavorable): 4 couleurs
  Z3 (optimum PROUVE)                  : 3 couleurs
```

### Lecture — glouton vs solveur, trois résultats a comparer

Trois resultats a comparer :
1. **Glouton bon ordre** : 3 couleurs (optimal par chance).
2. **Glouton mauvais ordre** : 4 couleurs (sous-optimal).
3. **Solveur Z3** : chi = 3 (optimal garanti).

### Pourquoi Petersen n'est pas 2-coloriable ?

Toute 2-coloration correspond a une **bipartition** : chaque arete relie un sommet du groupe 0 au groupe 1. Le graphe de Petersen contient un **triangle** (clique de taille 3) ? Non -- il ne contient pas de K3. Mais il contient un **cycle de longueur 5** dans le pentagone externe. Un cycle impair n'est pas 2-colorisable. Donc Petersen n'est pas 2-colorisable.

### Theoreme des 4 couleurs

Tout graphe planaire est 4-coloriable (Appel et Haken 1977, preuve assistee par ordinateur). Le graphe de Petersen n'est **pas planaire** (contient K_{3,3} comme mineur), donc le theoreme ne s'applique pas. Mais chi(Petersen) = 3 reste.

In [6]:
// Extraction et verification de la coloration optimale
use z3::{ast::{Ast, Bool, Int}, SatResult, Solver};

const N: usize = 10;

// Graphe de Petersen : 10 sommets (0..9), 15 aretes
let edges: Vec<(usize, usize)> = vec![
    // Pentagone externe
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
    // Rayons externe -> interne
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
    // Pentagramme interne
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),
];

// Liste d'adjacence (pour le glouton)
let mut adj: Vec<Vec<usize>> = vec![Vec::new(); N];
for (a, b) in &edges {
    adj[*a].push(*b);
    adj[*b].push(*a);
}

fn display_coloring(color: &[usize], title: &str, edges: &[(usize, usize)]) {
    println!("{}", title);
    let k = color.iter().copied().max().map_or(0, |m| m + 1);
    println!("  Couleurs utilisees : {}", k);
    for (v, c) in color.iter().enumerate() {
        println!("  Sommet {} -> couleur {}", v, c);
    }
    let valid = edges.iter().all(|(a, b)| color[*a] != color[*b]);
    println!("  Coloration valide (aucune arete monochrome) : {}", if valid { "True" } else { "False" });
}

fn greedy_coloring(n: usize, adjacency: &[Vec<usize>], order: &[usize]) -> Vec<i64> {
    let mut color: Vec<i64> = vec![-1; n];
    let mut forbidden_at: Vec<usize> = vec![0; n];
    for (step, &v) in order.iter().enumerate() {
        let stamp = step + 1;
        for &w in &adjacency[v] {
            if color[w] >= 0 {
                forbidden_at[color[w] as usize] = stamp;
            }
        }
        let mut c: i64 = 0;
        while forbidden_at[c as usize] == stamp {
            c += 1;
        }
        color[v] = c;
    }
    color
}

// Glouton first-fit, ordre naturel 0..9
let natural_order: Vec<usize> = (0..N).collect();
let greedy_natural: Vec<i64> = greedy_coloring(N, &adj, &natural_order);
let greedy_natural_colors: i64 = greedy_natural.iter().copied().max().map_or(0, |m| m + 1);

// Glouton first-fit, ordre defavorable (force la 4e couleur)
let bad_order: Vec<usize> = vec![4, 8, 2, 6, 5, 9, 0, 7, 1, 3];
let greedy_bad: Vec<i64> = greedy_coloring(N, &adj, &bad_order);
let greedy_bad_colors: i64 = greedy_bad.iter().copied().max().map_or(0, |m| m + 1);

// Modele Z3 et recherche lineaire du nombre chromatique
let color: Vec<Int> = (0..N).map(|v| Int::new_const(format!("C{v}"))).collect();
let mut optimal_color: Option<Vec<usize>> = None;
let mut chromatic: Option<usize> = None;
let mut proof_complete = true;
let mut lower_k_unsat = true;

for k in 1..=N {
    let k_i64 = k as i64;
    let s = Solver::new();
    for v in 0..N {
        s.assert(&color[v].ge(&Int::from_i64(0)));
        s.assert(&color[v].lt(&Int::from_i64(k_i64)));
    }
    s.assert(&color[0].eq(&Int::from_i64(0)));
    for (a, b) in &edges {
        s.assert(&color[*a].ne(&color[*b]));
    }

    match s.check() {
        SatResult::Unsat => {}
        SatResult::Unknown => {
            println!("k = {} -> UNKNOWN : Z3 n'a pas decide, optimalite non prouvee.", k);
            proof_complete = false;
            lower_k_unsat = false;
            break;
        }
        SatResult::Sat => {
            let m = match s.get_model() {
                Some(m) => m,
                None => {
                    println!("k = {} -> SAT mais aucun modele disponible : comparaison interrompue.", k);
                    proof_complete = false;
                    lower_k_unsat = false;
                    break;
                }
            };

            let mut sol: Vec<usize> = Vec::with_capacity(N);
            let mut complete = true;
            for v in 0..N {
                match m.eval(&color[v], true).and_then(|val| val.as_i64()) {
                    Some(c) => sol.push(c as usize),
                    None => {
                        complete = false;
                        break;
                    }
                }
            }
            if !complete {
                println!("Modele incomplet : coloration optimale non extraite, comparaison interrompue.");
                proof_complete = false;
                lower_k_unsat = false;
                break;
            }

            optimal_color = Some(sol);
            chromatic = Some(k);
            break;
        }
    }
}

match (chromatic, &optimal_color) {
    (Some(chromatic), Some(col)) if proof_complete && lower_k_unsat => {
        let title = format!("Coloration optimale Z3 (chi = {}) :", chromatic);
        display_coloring(col, &title, &edges);
        println!();
        println!("=== Comparaison ===");
        println!("  Glouton first-fit (ordre naturel)    : {} couleurs", greedy_natural_colors);
        println!("  Glouton first-fit (ordre defavorable): {} couleurs", greedy_bad_colors);
        println!("  Z3 (optimum PROUVE)                  : {} couleurs", chromatic);
        println!("  Dans le pire ordre, le glouton gaspille {} couleur(s).", greedy_bad_colors - chromatic as i64);
        println!("  Surtout : seul Z3 PROUVE que {} est minimal (2 couleurs = UNSAT).", chromatic);
    }
    _ => {
        println!("Coloration optimale indisponible : aucune conclusion sur l'optimum n'est affichee.");
    }
};

Coloration optimale Z3 (chi = 3) :


  Couleurs utilisees : 3


  Sommet 0 -> couleur 0


  Sommet 1 -> couleur 1


  Sommet 2 -> couleur 2


  Sommet 3 -> couleur 0


... (9 outputs left out)

  Glouton first-fit (ordre defavorable): 4 couleurs


  Z3 (optimum PROUVE)                  : 3 couleurs


  Dans le pire ordre, le glouton gaspille 1 couleur(s).


  Surtout : seul Z3 PROUVE que 3 est minimal (2 couleurs = UNSAT).


## Visualisation : la coloration optimale

Le graphe de Petersen se dessine classiquement avec un **pentagone externe** (sommets 0-4) et un **pentagramme interne** (sommets 5-9), relies par 5 rayons (0-5, 1-6, ..., 4-9).

### Verbatim code[6]

Dans ce carnet Rust, la visualisation matplotlib du jumeau Python n'est **pas portee** : aucune figure n'est produite. La cellule est reecrite en un **rendu texte** de la coloration -- chaque sommet est affiche ligne a ligne avec sa position canonique et sa couleur.

### Couleurs dans la 3-coloration

La répartition dépend du modèle choisi par Z3, car le Petersen possède plusieurs 3-colorations. Dans le témoin affiché dans la cellule de rendu :

- **Couleur 0** : sommets 0, 3, 6 et 7.
- **Couleur 1** : sommets 1, 4 et 5.
- **Couleur 2** : sommets 2, 8 et 9.

Un autre modèle peut répartir les sommets différemment tout en restant valide.

### Implementation (rendu texte)

```rust
// Positions canoniques : pentagone externe (rayon 2) + pentagramme interne (rayon 1)
let mut pos: Vec<(f64, f64)> = vec![(0.0, 0.0); N];
for i in 0..5 {
    let ang = std::f64::consts::PI / 2.0 + 2.0 * std::f64::consts::PI * (i as f64) / 5.0;
    pos[i] = (2.0 * ang.cos(), 2.0 * ang.sin());        // externe (rayon 2)
    let ang2 = ang + std::f64::consts::PI / 5.0;          // interne decale
    pos[5 + i] = (1.0 * ang2.cos(), 1.0 * ang2.sin());     // interne (rayon 1)
}
// Palette : rouge / bleu / vert
let palette = ["rouge", "bleu", "vert"];
for v in 0..N {
    println!("  Sommet {} : position ({:.3}, {:.3}) -> couleur {} ({})",
             v, pos[v].0, pos[v].1, optimal_color[v], palette[optimal_color[v]]);
}
```

### Symetrie des couleurs

Les six permutations des trois couleurs forment le groupe symetrique `S_3`. Une permutation de ces couleurs produit une autre 3-coloration valide. Le sous-groupe des trois permutations cycliques est `Z_3`, mais il ne décrit pas toutes les permutations possibles. Z3 énumère 120 colorations distinctes (avant identification par permutation) -- cf Exercice 3.

In [7]:
// Visualisation de la coloration 3-chromatique du graphe de Petersen
// Rendu texte : aucune figure n'est produite (matplotlib n'est pas porte dans ce lot)
use z3::{ast::{Ast, Bool, Int}, SatResult, Solver};

const N: usize = 10;

// Graphe de Petersen : 10 sommets (0..9), 15 aretes
let edges: Vec<(usize, usize)> = vec![
    // Pentagone externe
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
    // Rayons externe -> interne
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
    // Pentagramme interne
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),
];

// Liste d'adjacence complete, reproduite pour que le graphe soit autonome
let mut adj: Vec<Vec<usize>> = vec![Vec::new(); N];
for (a, b) in &edges {
    adj[*a].push(*b);
    adj[*b].push(*a);
}
let _ = &adj;

// Modele Z3 et recherche du nombre chromatique
let color: Vec<Int> = (0..N).map(|v| Int::new_const(format!("C{v}"))).collect();
let mut optimal_color: Option<Vec<usize>> = None;
let mut chromatic: Option<usize> = None;
let mut proof_complete = true;

for k in 1..=N {
    let k_i64 = k as i64;
    let s = Solver::new();
    for v in 0..N {
        s.assert(&color[v].ge(&Int::from_i64(0)));
        s.assert(&color[v].lt(&Int::from_i64(k_i64)));
    }
    s.assert(&color[0].eq(&Int::from_i64(0)));
    for (a, b) in &edges {
        s.assert(&color[*a].ne(&color[*b]));
    }

    match s.check() {
        SatResult::Unsat => {}
        SatResult::Unknown => {
            println!("k = {} -> UNKNOWN : Z3 n'a pas decide, optimalite non prouvee.", k);
            proof_complete = false;
            break;
        }
        SatResult::Sat => {
            let m = match s.get_model() {
                Some(m) => m,
                None => {
                    println!("k = {} -> SAT mais aucun modele disponible : rendu de la coloration interrompue.", k);
                    proof_complete = false;
                    break;
                }
            };

            let mut sol: Vec<usize> = Vec::with_capacity(N);
            let mut complete = true;
            for v in 0..N {
                match m.eval(&color[v], true).and_then(|val| val.as_i64()) {
                    Some(c) => sol.push(c as usize),
                    None => {
                        complete = false;
                        break;
                    }
                }
            }
            if !complete {
                println!("Modele incomplet : rendu de la coloration interrompu.");
                proof_complete = false;
                break;
            }

            optimal_color = Some(sol);
            chromatic = Some(k);
            break;
        }
    }
}

// Positions canoniques : pentagone externe + pentagramme interne
let mut pos: Vec<(f64, f64)> = vec![(0.0, 0.0); N];
for i in 0..5 {
    let ang = std::f64::consts::PI / 2.0 + 2.0 * std::f64::consts::PI * (i as f64) / 5.0;
    // externe (rayon 2)
    pos[i] = (2.0 * ang.cos(), 2.0 * ang.sin());
    let ang2 = ang + std::f64::consts::PI / 5.0; // interne decale
    // interne (rayon 1)
    pos[5 + i] = (1.0 * ang2.cos(), 1.0 * ang2.sin());
}

// Palette : rouge / bleu / vert
let palette = ["rouge", "bleu", "vert"];

match (proof_complete, chromatic, &optimal_color) {
    (true, Some(chromatic), Some(col)) => {
        println!("Petersen : coloration optimale chi = {}", chromatic);
        for v in 0..N {
            let name = if (*col)[v] < palette.len() { palette[(*col)[v]] } else { "?" };
            println!("  Sommet {} : position ({:.3}, {:.3}) -> couleur {} ({})", v, pos[v].0, pos[v].1, (*col)[v], name);
        }
        println!("  Aretes ({}):", edges.len());
        for (a, b) in &edges {
            println!("    ({}, {}) : couleur {} vs couleur {}", a, b, (*col)[*a], (*col)[*b]);
        }
    }
    _ => {
        println!("Coloration optimale indisponible : aucun titre d'optimalite affiche.");
    }
};

Petersen : coloration optimale chi = 3


  Sommet 0 : position (0.000, 2.000) -> couleur 0 (rouge)


  Sommet 1 : position (-1.902, 0.618) -> couleur 1 (bleu)


  Sommet 2 : position (-1.176, -1.618) -> couleur 2 (vert)


  Sommet 3 : position (1.176, -1.618) -> couleur 0 (rouge)


  Sommet 4 : position (1.902, 0.618) -> couleur 1 (bleu)


... (17 outputs left out)

    (7, 9) : couleur 0 vs couleur 2


    (9, 6) : couleur 2 vs couleur 0


    (6, 8) : couleur 0 vs couleur 2


    (8, 5) : couleur 2 vs couleur 1


### Verbatim code[6] : rendu texte de la coloration

Dans ce carnet Rust, la visualisation matplotlib du jumeau Python n'est **pas portee** : aucune figure n'est produite. La cellule de code correspondante est reecrite en un **rendu texte** de la coloration -- une ligne par sommet (position canonique + couleur) et une ligne par arete (couleur aux deux extremites).

### Lecture approfondie

Le code utilise des positions canoniques :
- **Pentagone externe** : 5 sommets sur un cercle de rayon **2.0**, angles 90 + 72*i degrés.
- **Pentagramme interne** : 5 sommets sur un cercle de rayon **1.0**, angles 90 + 72*i + 36 degrés (rotation de 36 degrés).

Chaque sommet est affiche avec la couleur determinee par la 3-coloration et la palette explicite du code : couleur 0 = **rouge**, couleur 1 = **bleu**, couleur 2 = **vert** (memes teintes que la palette matplotlib du jumeau Python).

### Cout computationnel

10 sommets, 15 aretes : un seul passage de rendu -- la sortie n'affiche pas de mesure de temps.

### Implementation alternative : networkx

```python
import networkx as nx
import matplotlib.pyplot as plt

G = nx.Graph()
G.add_nodes_from(range(N))
G.add_edges_from(edges)
pos = nx.shell_layout(G, nlist=[range(5), range(5, 10)])
nx.draw(G, pos, node_color=[f'C{color[v]}' for v in range(N)], with_labels=True, node_size=500)
```

Cette alternative n'est **pas portee** dans ce lot : aucune crate de tracé n'est utilisee, le carnet reste en rendu texte. Pour information, elle est plus concise, mais moins de contrôle sur les positions.

### Intérêt pédagogique

Le rendu permet de voir immediatement la structure du Petersen : un cycle externe et un cycle interne (pentagramme) relies par 5 rayons. La 3-coloration est evidente a la lecture.

## 5. Glouton vs solveur : ce que demontre Petersen

Le graphe de Petersen illustre **trois leçons de pedagogie** :

### Lecon 1 : L'heuristique peut echouer

Le glouton first-fit peut donner **4 couleurs** au lieu de 3 (optimal). Cela depend de l'ordre de parcours. Pour des graphes industriels (coloration de cartes geographiques avec contraintes), le glouton sous-optimal peut-être inacceptable.

### Lecon 2 : Le solveur prouve l'optimalite

Z3 fournit une **preuve** que chi = 3 : k=1 (UNSAT) et k=2 (UNSAT) sont démontrées mathématiquement. Le solveur utilise la **propagation de clauses** et la **théorie QF_LIA** (linear integer arithmetic) pour eliminer les cas impossibles.

### Lecon 3 : Le cout n'est pas toujours proportionnel a la taille

Pour certains graphes (Mycielski, Kneser), la recherche linéaire peut prendre **plusieurs secondes** même pour n < 50 sommets. Pour d'autres graphes (planaires, grands), le solveur peut trouver l'optimal en quelques ms grace aux structures speciales (théorème des 4 couleurs pour les planaires).

### Verbatim code[5] : comparaison directe

Sortie verbatim : `Glouton bon ordre = 3 couleurs ; Glouton mauvais ordre = 4 couleurs ; Z3 optimal = chi = 3`.

### Stratégie hybride

En pratique, on peut utiliser :
1. **Glouton rapide** pour avoir une borne superieure (rapide, parfois optimale).
2. **Solveur SMT** pour trouver l'optimum ou prouver la borne superieure.
3. **Recherche binaire** sur k entre la borne du glouton et N.

## Exercices

Les trois exercices ci-dessous **generalisent la coloration** a des variantes : forcer une clique de taille 4 (pour augmenter chi), planification d'examens (coloration deguisee), comptage des colorations distinctes (enumeration SMT).

### Convention Exemple vs Exercice (règle C.1)

- **Exemples guides** = code résolu fonctionnel (NE PAS stubber, NE PAS relabeler en exercice).
- **Exercices** = stub avec `// TODO etudiant :` (NE PAS remplir la solution).
- Reference : `exercise-example-labeling.md` (mandat user 2026-05-20, anti-pendule).

### Vue d'ensemble

| Cellule | Type | Contenu |
|---------|------|---------|
| code[7] | Exercice 1 | Forcer une quatrième couleur (clique K4) |
| code[8] | Exercice 2 | Planification d'examens (coloration deguisee) |
| code[9] | Exercice 3 | Compter les 3-colorations distinctes |

### Pattern commun

```rust
// 1. Definir le graphe
// 2. Encoder en SMT
let color: Vec<Int> = (0..N).map(|v| Int::new_const(format!("C{v}"))).collect();
let s = Solver::new();
for v in 0..N {
    s.assert(&color[v].ge(&Int::from_i64(0)));
    s.assert(&color[v].lt(&Int::from_i64(k as i64)));
}
for (u, v) in &edges {
    s.assert(&color[*u].ne(&color[*v]));
}
// 3. Verifier ou enumerer
if s.check() == SatResult::Sat {
    println!("{:?}", s.get_model()); // Option<Model> : aucun unwrap() ici
}
```

### Exercice 1 - Forcer une quatrième couleur

Le graphe de Petersen est **sans triangles** (maille 5) : 4 de ses sommets portent au plus 3 aretes, et un K4 en exige 6 -- ajouter **une seule** arete ne peut donc pas y créer une clique de taille 4. Ajoutez le **nombre d'aretes necessaire** pour créer une **clique de taille 4** (K4). Recalculez `chi` et verifiez que le solveur Z3 retourne `chi = 4` au lieu de `chi = 3`.

### Pattern attendu

```rust
// Choisir 4 sommets formant une clique (toutes les paires sont deja reliees)
// Le graphe de Petersen n'a pas de K4 natif. Il faut ajouter 3 aretes pour completer.
// Par exemple, sommets {0, 1, 5, 6} : on a 0-1 et 0-5 et 1-6 deja. Il manque 5-6 et 5-1 et 6-0.
// Ajouter 5-6, 0-6, 1-5 : on obtient une K4 sur {0, 1, 5, 6}.

let mut edges_modified: Vec<(usize, usize)> = edges.clone();
edges_modified.extend(vec![(5, 6), (0, 6), (1, 5)]);

// Relancer la recherche lineaire sur edges_modified
// Verifier que k=3 devient UNSAT et k=4 devient SAT
```

### Verdict attendu

`chi = 4` -- la presence d'un K4 force au moins 4 couleurs (tous les sommets du K4 doivent avoir des couleurs différentes).

### Cout computationnel

Recherche linéaire : k=1 (UNSAT), k=2 (UNSAT), k=3 (UNSAT, plus long car preuve), k=4 (SAT).

### Pour aller plus loin

Generalisation : pour tout graphe G, `chi(G) >= omega(G)` (taille de la plus grande clique). Pour Petersen modifie, omega = 4 et chi = 4 -- optimal.

> **Indice :**

Vérifier que 5-6, 0-6 et 1-5 ne sont pas déjà des aretes du Petersen (elles ne le sont pas : le pentagramme interne relie 5-7, 7-9, 9-6, 6-8, 8-5).

In [8]:
// Exercice 1 : ajouter une arete creant une clique de taille 4, recalculer chi
// TODO etudiant :
// Etape 1 : choisir 4 sommets et lister les aretes a ajouter pour qu'ils forment une clique
// Etape 2 : construire un nouveau modele avec les aretes supplementaires
// Etape 3 : relancer la recherche lineaire et verifier chi == 4
let mut result: Option<usize> = None; // TODO etudiant
println!("Exercice 1 a completer : forcez chi = 4 en ajoutant une clique de taille 4");

Exercice 1 a completer : forcez chi = 4 en ajoutant une clique de taille 4


### Exercice 2 - Planification d'examens

La planification d'examens est une **coloration deguisee** : sommets = examens, aretes = étudiants communs, couleurs = créneaux horaires.

### Pattern attendu

```rust
// Definir un graphe de conflits
let examens = ["Algo", "BD", "IA", "Stats", "ML"];
// Etudiants inscrits a plusieurs examens
// Nombre d'etudiants par paire d'examens :
// Algo+BD (3), Algo+IA (2), Algo+Stats (4), Algo+ML (2)
// BD+IA (3), BD+Stats (2), BD+ML (1)
// IA+Stats (2), IA+ML (3)
// Stats+ML (1)
let edges_conflicts: Vec<(usize, usize)> =
    vec![(0,1),(0,2),(0,3),(0,4),(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)];
// (toutes les paires -- graphe complet K5)

// Le solveur trouve chi = 5 (il faut 5 creneaux)
```

### Verdict attendu

`chi(K_5) = 5` -- un graphe complet K_n necessite n couleurs (toutes les paires sont différentes).

### Cout computationnel

Recherche linéaire : k=1 (UNSAT), k=2 (UNSAT), ..., k=4 (UNSAT), k=5 (SAT). Pour K_5, c'est rapide.

### Pour aller plus loin

Cas reel : 200 examens, 5000 étudiants, degré moyen 12. Le solveur Z3 trouve le nombre minimum de créneaux en quelques secondes.

> **Indice :**

Construire un graphe avec 5 sommets et toutes les paires (10 aretes = K_5).

In [9]:
// Exercice 2 : planification d'examens (coloration deguisee)
// TODO etudiant :
// Etape 1 : definir votre graphe de conflits (sommets = examens, aretes = etudiants communs)
// Etape 2 : trouver le nombre minimal de creneaux via la recherche lineaire
// Etape 3 : afficher l'emploi du temps par creneau
let mut result: Option<Vec<Vec<usize>>> = None; // TODO etudiant
println!("Exercice 2 a completer : planifiez les examens par coloration de graphe");

Exercice 2 a completer : planifiez les examens par coloration de graphe


### Exercice 3 - Compter les 3-colorations

Combien de colorations distinctes a 3 couleurs admet le graphe de Petersen ?

### Verdict attendu

Le graphe de Petersen admet **120 colorations distinctes** à 3 couleurs ; à permutation des 3 couleurs près (3! = 6), cela fait **120 / 6 = 20 classes** — le stub attend `120 total, 20 a permutation pres`.

### Cout computationnel

Z3 énumère les 120 solutions en quelques secondes. Pour des graphes plus gros, l'enumeration peut prendre plusieurs minutes.

### Pour aller plus loin

Le **polynome chromatique** P(G, k) donne le nombre exact de k-colorations pour chaque k. Pour Petersen, P(G, k) = k(k-1)(k-2)(k^7 - 12k^6 + 67k^5 - 230k^4 + 529k^3 - 814k^2 + 775k - 352). Vérifie : P(G, 3) = 3 x 2 x 1 x 20 = **120**, conforme au compte attendu.

> **Indice:**

La construction de la disjonction peut se faire en Rust avec une boucle qui conserve une valeur du modèle pour chaque sommet, puis exclut exactement cette solution :

```rust
let mut disjuncts: Vec<Bool> = Vec::new();
for v in 0..N {
    let valeur = /* valeur entiere de color[v] dans le modele */;
    disjuncts.push(color[v].ne(&valeur));
}
let refs: Vec<&Bool> = disjuncts.iter().collect();
s.assert(&Bool::or(&refs));
```

La disjonction exclut la solution trouvee, et elle seule. Les 6 permutations d'une même coloration sont énumérées séparément, d'où la division finale par 3! = 6.

### Verification

Recoupement : le polynome chromatique donne P(G, 3) = 120 (cf Pour aller plus loin) -- le compte exhaustif doit tomber sur le même total.

In [10]:
// Exercice 3 : compter les 3-colorations en enumerant les solutions
// TODO etudiant :
// Etape 1 : boucle qui resout, puis exclut la solution trouvee, jusqu'a UNSAT
// Etape 2 : compter les solutions ; comparer a 120
// Etape 3 : diviser par 3! = 6 pour obtenir les colorations a permutation pres
let mut nb_colorations: Option<usize> = None; // TODO etudiant
println!("Exercice 3 a completer : enumerez les 3-colorations (attendu : 120 total, 20 a permutation pres)");

Exercice 3 a completer : enumerez les 3-colorations (attendu : 120 total, 20 a permutation pres)


In [11]:
// ### Exemple guidé – Exercice 3 (à consulter après votre tentative)
//
// Énumération réelle des 3-colorations du graphe de Petersen : chaque solution
// SAT trouvée est exclue par une disjonction, jusqu'à UNSAT. Le total attendu
// (120 colorations, 20 classes à permutation près) est ancré dans la sortie.

use z3::{ast::{Ast, Bool, Int}, SatResult, Solver};

const N: usize = 10;

// Graphe de Petersen : 10 sommets (0..9), 15 aretes
let edges: Vec<(usize, usize)> = vec![
    // Pentagone externe
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),
    // Rayons externe -> interne
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),
    // Pentagramme interne
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),
];

// Une variable entiere de couleur par sommet, nommees C0..C9
let color: Vec<Int> = (0..N).map(|v| Int::new_const(format!("C{v}"))).collect();

let s = Solver::new();
for v in 0..N {
    s.assert(&color[v].ge(&Int::from_i64(0)));
    s.assert(&color[v].lt(&Int::from_i64(3)));
}
for (a, b) in &edges {
    s.assert(&color[*a].ne(&color[*b]));
}

let mut nb_colorations: usize = 0;
let mut enumeration_complete = false;

loop {
    match s.check() {
        SatResult::Sat => {}
        SatResult::Unsat => {
            enumeration_complete = true;
            break;
        }
        SatResult::Unknown => {
            println!("Enumeration interrompue : Z3 a retourne UNKNOWN.");
            break;
        }
    }

    let m = match s.get_model() {
        Some(m) => m,
        None => {
            println!("Enumeration interrompue : aucun modele disponible.");
            break;
        }
    };

    let mut disjuncts: Vec<Bool> = Vec::new();
    let mut extraction_complete = true;
    for v in 0..N {
        match m.eval(&color[v], true) {
            Some(val) => disjuncts.push(color[v].ne(&val)),
            None => {
                extraction_complete = false;
                break;
            }
        }
    }

    if !extraction_complete {
        println!("Enumeration interrompue : le modele ne fournit pas toutes les couleurs.");
        break;
    }

    // La disjonction exclut exactement la solution complete trouvee.
    let refs: Vec<&Bool> = disjuncts.iter().collect();
    s.assert(&Bool::or(&refs));
    nb_colorations += 1;
}

if enumeration_complete {
    println!("Nombre de 3-colorations : {}", nb_colorations);
    println!("A permutation pres (3! = 6) : {} classes", nb_colorations / 6);
} else {
    println!("Aucun total exhaustif affiche : l'enumeration n'a pas atteint UNSAT.");
}

// Verdict : 120 colorations distinctes, soit 120 / 6 = 20 classes a permutation
// des 3 couleurs pres -- conforme au stub (attendu : 120 total, 20 a permutation pres).


Nombre de 3-colorations : 120


A permutation pres (3! = 6) : 20 classes


()

## Conclusion

La coloration de graphe cristallise la différence entre **heuristique** et **preuve** :

1. **Le glouton first-fit** est ultra-rapide (O(n+m)) mais peut-être sous-optimal. Pour Petersen, il donne 3 ou 4 couleurs selon l'ordre.
2. **Le solveur Z3** prouve l'optimalite : `chi(Petersen) = 3` est demontre mathematiquement (k=1, 2 sont UNSAT).
3. **Le graphe de Petersen** est un benchmark classique : non-hamiltonien, non-planaire, 3-regulier, 120 automorphismes. Il sert de test pour les algorithmes de coloration.

### Ce que ce notebook demontre

1. **Modèle Z3** : 1 variable entière par sommet + contraintes `!=`.
2. **Recherche linéaire** : k=1..N pour trouver chi(G).
3. **Preuve d'optimalite** : Z3 fournit SAT/UNSAT pour chaque k.
4. **Glouton vs solveur** : 3 leçons (heuristique peut echouer, solveur prouve, cout variable).
5. **Enumeration** : compter toutes les colorations distinctes avec `while s.check() == SatResult::Sat`.
6. **Generalisation** : planification d'examens (coloration deguisee), K_n necessite n couleurs.

### Pour aller plus loin

- **DSATUR** (Daniel Brélaz 1979) : heuristique qui colore en priorisant les sommets satures. Souvent optimale.
- **Welsh-Powell** : variante du glouton qui ordonne par degré decroissant. Optimal pour les graphes d'intervalles.
- **OR-Tools CP-SAT** : solveur specialise pour les problemes combinatoires (cf CSP-8-Temporal).
- **Polynome chromatique** : formule exacte de P(G, k), calculable par deletion-contraction.
- **Théorème des 4 couleurs** (Appel-Haken 1977) : tout graphe planaire est 4-coloriable. La preuve a ete partiellement assistee par ordinateur.

### References

- Petersen 1898 *Sur le théorème de Tait*. L'Intermédiaire des Mathématiciens 5: 225-227.
- Karp 1972 *Reducibility among combinatorial problems*. Complexity of Computer Computations: 85-103 (NP-completude de 3-COLOR).
- Brélaz 1979 *New methods to color the vertices of a graph*. Communications of the ACM 22(4): 251-256 (DSATUR).
- Welsh, Powell 1967 *An upper bound for the chromatic number of a graph and its application to timetabling problems*. The Computer Journal 10(1): 85-86.
- de Moura, Bjorner 2008 *Z3: An Efficient SMT Solver*. TACAS 2008.